In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.utils import murmurhash3_32

from tiger.config import CODEBOOK_SIZE, MIN_REVIEWS_PER_USER, NUM_CODEBOOKS, SEQUENCE_LENGTH, USER_NUM_BINS

In [2]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 2070


In [3]:
seq = pd.read_parquet("./data/interim/2014/sequences_Beauty.parquet")
sem = pd.read_parquet("./data/processed/2014/Beauty_semantic_ids.parquet")
items = pd.read_parquet("./data/interim/2014/meta_Beauty.parquet")

In [4]:
asin_to_idx = {asin: i for i, asin in enumerate(items["asin"].tolist())}

idx_to_sem_id = {}
for idx, raw_sem_id in enumerate(sem.to_dict(orient="tight")["data"]):
    proc_sem_id = [level * CODEBOOK_SIZE + sem_token for level, sem_token in enumerate(raw_sem_id)]
    idx_to_sem_id[idx] = proc_sem_id

asin_to_sem_id = {asin: idx_to_sem_id[asin_to_idx[asin]] for asin in list(asin_to_idx.keys())}

In [5]:
def to_semantic_sequence(user_id: str, asin_seq: list[str], num_bins: int):
    offset = (NUM_CODEBOOKS + 1) * CODEBOOK_SIZE
    user_token_id = murmurhash3_32(user_id, positive=True) % num_bins + offset

    sem_seq: np.ndarray = np.array([user_token_id], dtype=np.int64)
    for asin in asin_seq:
        sem_id = asin_to_sem_id[asin]
        sem_seq = np.append(sem_seq, sem_id)
    return sem_seq

In [6]:
sem_seqs = seq.apply(lambda x: to_semantic_sequence(x["reviewerID"], x["asin"], USER_NUM_BINS), axis=1)

In [7]:
with open("./data/processed/2014/Beauty_semantic_sequences.npy", "wb") as f:
    np.save(f, sem_seqs)

In [8]:
min_len = float("inf")
max_len = -float("inf")
for sem_seq in sem_seqs:
    min_len = min(sem_seq.shape[0], min_len)
    max_len = max(sem_seq.shape[0], max_len)

print(f"Sequences lengths are in the range [{min_len}, {max_len}]")

assert min_len == (NUM_CODEBOOKS + 1) * (MIN_REVIEWS_PER_USER) + 1
assert max_len == (NUM_CODEBOOKS + 1) * SEQUENCE_LENGTH + 1

Sequences lengths are in the range [21, 81]
